In [ ]:
# =============================================================================
# Deep Reinforcement Learning for Structural Damage Detection
# Algorithm: Twin Delayed Deep Deterministic Policy Gradient (TD3)
# =============================================================================
#
# Research Background
# -------------------
# This notebook implements a reinforcement learning framework for structural
# damage detection in a multi-story shear building model. The goal is to
# estimate story-wise damage indices from measured structural response data.
#
# Problem Definition
# ------------------
# Input:
#   Acceleration response signals recorded from multiple floors
#
# Output:
#   A continuous damage vector representing the damage severity at each story
#
# Reinforcement Learning Formulation
# ----------------------------------
# Observation:
#   Flattened and normalized acceleration time-series data
#
# Action:
#   Continuous-valued damage coefficients for each floor
#   with bounds in the interval [0.5, 1.0]
#
# Reward:
#   Negative mean absolute error between predicted and true damage values
#
# Motivation for TD3
# ------------------
# TD3 is an actor-critic algorithm designed for continuous control problems.
# It improves over DDPG by using:
#   1. Clipped double Q-learning
#   2. Delayed policy updates
#   3. Target policy smoothing
#
# These mechanisms help reduce overestimation bias and improve training
# stability, which is particularly important when estimating continuous
# structural damage parameters.
#
# Dataset Structure
# -----------------
# RL_Dataset.zip
#   ├── labels.json
#   ├── scenario_1.csv
#   ├── scenario_2.csv
#   └── ...
#
# Each CSV file contains structural response data for one scenario, and
# labels.json stores the corresponding damage pattern.
#
# Notes
# -----
# This implementation is written in a clean and reproducible format for:
#   - academic reporting
#   - public GitHub repositories
#   - technical portfolios
#   - benchmark comparison with classical methods and other RL algorithms
# =============================================================================



# =============================================================================
# 1. Install Required Libraries (for Google Colab)
# =============================================================================
!pip install stable-baselines3[extra] shimmy scikit-learn --quiet



# =============================================================================
# 2. Import Libraries
# =============================================================================
import os
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

import gymnasium as gym
from gymnasium import spaces

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

from stable_baselines3 import TD3
from stable_baselines3.common.callbacks import CheckpointCallback
from stable_baselines3.common.noise import NormalActionNoise



# =============================================================================
# 3. Upload and Extract Dataset
# =============================================================================
from google.colab import files

uploaded = files.upload()

with zipfile.ZipFile("RL_Dataset.zip", "r") as zip_ref:
    zip_ref.extractall("/content/RL_Dataset")

dataset_dir = "/content/RL_Dataset"

print("Dataset extracted successfully.")



# =============================================================================
# 4. Load Labels and Initialize Scaler
# =============================================================================
labels_path = os.path.join(dataset_dir, "labels.json")

with open(labels_path, "r") as f:
    labels_data = json.load(f)

scaler = MinMaxScaler(feature_range=(-1, 1))

print(f"Number of scenarios loaded: {len(labels_data)}")



# =============================================================================
# 5. Define Custom Reinforcement Learning Environment
# =============================================================================
class DamageDetectionEnv(gym.Env):
    """
    Custom reinforcement learning environment for structural damage detection.

    The environment randomly selects one structural scenario at each reset.
    The observation is a normalized acceleration time-series vector, and
    the agent predicts a continuous damage vector for all floors.

    Observation:
        Flattened normalized time-series data from all floors

    Action:
        Continuous damage values in the range [0.5, 1.0]

    Reward:
        Negative mean absolute error between predicted and true damage
    """

    metadata = {"render_modes": []}

    def __init__(self, dataset_dir, labels):
        super().__init__()

        self.dataset_dir = dataset_dir
        self.labels = labels

        self.num_floors = 5
        self.num_samples = 100

        self.current_observation = None
        self.current_label = None

        self.observation_space = spaces.Box(
            low=-1.0,
            high=1.0,
            shape=(self.num_floors * self.num_samples,),
            dtype=np.float32
        )

        self.action_space = spaces.Box(
            low=0.5,
            high=1.0,
            shape=(self.num_floors,),
            dtype=np.float32
        )

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        idx = np.random.randint(0, len(self.labels))
        label_info = self.labels[idx]

        file_path = os.path.join(
            self.dataset_dir,
            f"{label_info['scenario']}.csv"
        )

        data = pd.read_csv(file_path).values[:self.num_samples, :]
        scaled_data = scaler.fit_transform(data)

        self.current_observation = scaled_data.flatten().astype(np.float32)
        self.current_label = np.array(label_info["damage_pattern"], dtype=np.float32)

        return self.current_observation, {}

    def step(self, action):
        action = np.array(action, dtype=np.float32)

        error = np.abs(action - self.current_label)
        reward = -np.mean(error)

        terminated = True
        truncated = False

        info = {
            "true_damage": self.current_label.tolist(),
            "predicted": action.tolist()
        }

        return self.current_observation, reward, terminated, truncated, info



# =============================================================================
# 6. Initialize Environment
# =============================================================================
env = DamageDetectionEnv(dataset_dir=dataset_dir, labels=labels_data)



# =============================================================================
# 7. Define Action Noise
# =============================================================================
n_actions = env.action_space.shape[-1]

action_noise = NormalActionNoise(
    mean=np.zeros(n_actions),
    sigma=0.05 * np.ones(n_actions)
)



# =============================================================================
# 8. Define Neural Network Architecture
# =============================================================================
policy_kwargs = dict(
    net_arch=[256, 256],
    activation_fn=torch.nn.ReLU
)



# =============================================================================
# 9. Create TD3 Model
# =============================================================================
model = TD3(
    policy="MlpPolicy",
    env=env,
    verbose=1,
    learning_rate=1e-4,
    policy_kwargs=policy_kwargs,
    action_noise=action_noise,
    batch_size=64,
    train_freq=1,
    gradient_steps=1,
    buffer_size=1_000_000,
    learning_starts=1000
)



# =============================================================================
# 10. Define Checkpoint Callback
# =============================================================================
checkpoint_callback = CheckpointCallback(
    save_freq=10000,
    save_path="./td3_checkpoints/",
    name_prefix="td3_model"
)



# =============================================================================
# 11. Train the Model
# =============================================================================
model.learn(
    total_timesteps=100000,
    callback=checkpoint_callback
)

print("Training completed successfully.")



# =============================================================================
# 12. Save the Trained Model
# =============================================================================
model.save("td3_damage_detector")

print("Model saved successfully.")



# =============================================================================
# 13. Evaluate the Trained Model
# =============================================================================
results = []

for _ in range(20):
    obs, _ = env.reset()

    action, _ = model.predict(obs, deterministic=True)

    _, _, _, _, info = env.step(action)

    results.append(info)

print("Evaluation completed successfully.")



# =============================================================================
# 14. Visualize Prediction Errors
# =============================================================================
true_vals = np.array([r["true_damage"] for r in results])
pred_vals = np.array([r["predicted"] for r in results])
errors = np.abs(true_vals - pred_vals)

plt.figure(figsize=(10, 5))
plt.boxplot(errors, tick_labels=[f"Floor {i+1}" for i in range(5)])
plt.title("Absolute Prediction Error of TD3 Model per Floor")
plt.ylabel("Absolute Error |Predicted - True|")
plt.grid(True)
plt.show()



# =============================================================================
# 15. Compute Quantitative Performance Metrics
# =============================================================================
y_true = true_vals.flatten()
y_pred = pred_vals.flatten()

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
accuracy = np.mean(np.abs(y_true - y_pred) < 0.05) * 100

print("TD3 Performance Metrics")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"Accuracy (|error| < 0.05): {accuracy:.2f}%")
